# DATA PREP FOR HOW AI IS SHAPING THE WORKFORCE

This notebook is for exploratory puruposes. If regular data refresh is required, move this into a structured scripted.

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

import json

## Load in BLS AI exposure data

In [2]:
bls_ai = pd.read_excel('ai-exposure-categories.xlsx', skiprows=1)
bls_ai = bls_ai.dropna(subset=['2025 National Employment Matrix code'])
# bls_ai.info(show_counts=True)
# cleaned_bls_ai = bls_ai.where(pd.notna(bls_ai), None).to_dict(orient='records')
occupation_df = pd.read_excel('occupation.xlsx', sheet_name='Table 1.10', skiprows=1)
occupation_df = occupation_df.dropna(subset=['2025 National Employment Matrix code'])
occupation_df = (occupation_df[
    [
        '2025 National Employment Matrix code',
        'Labor force exit rate, 2025–35 annual average',
        'Occupational transfer rate, 2025–35 annual average',
        'Total occupational separations rate, 2025–35 annual average',
        'Labor force exits, 2025–35 annual average',
        'Occupational transfers, 2025–35 annual average',
        'Total occupational separations, 2025–35 annual average',
        'Occupational openings, 2025–35 annual average'
    ]
])
# occupation_df.info(show_counts=True)

joined_df = pd.merge(bls_ai, occupation_df, how='left', on='2025 National Employment Matrix code')


joined_df = joined_df.rename(columns={
    '2025 National Employment Matrix code': 'SOC code',
    '2025 National Employment Matrix title': 'Occupation title',
    'Employment change, percent, 2025–35': 'Employment change, percent',
    'Labor force exit rate, 2025–35 annual average': 'Labor force exit rate, annual average',
    'Occupational transfer rate, 2025–35 annual average': 'Occupational transfer rate, annual average',
    'Total occupational separations rate, 2025–35 annual average': 'Total occupational separations rate, annual average',
    'Labor force exits, 2025–35 annual average': 'Labor force exits, annual average',
    'Occupational transfers, 2025–35 annual average': 'Occupational transfers, annual average',
    'Total occupational separations, 2025–35 annual average': 'Total occupational separations, annual average',
    'Occupational openings, 2025–35 annual average': 'Occupational openings, annual average'
})
joined_df['Industry code'] = joined_df['SOC code'].str.split('-',n=1).str[0]
joined_df.info(show_counts=True)
# cleaned_occupation_df = occupation_df.where(pd.notna(occupation_df), None).to_dict(orient='records')

<class 'pandas.DataFrame'>
RangeIndex: 831 entries, 0 to 830
Data columns (total 21 columns):
 #   Column                                                                     Non-Null Count  Dtype  
---  ------                                                                     --------------  -----  
 0   Occupation title                                                           831 non-null    str    
 1   SOC code                                                                   831 non-null    str    
 2   Employment, 2025                                                           831 non-null    float64
 3   Employment, 2035                                                           831 non-null    float64
 4   Employment change, numeric, 2025–35                                        831 non-null    float64
 5   Employment change, percent                                                 831 non-null    float64
 6   Percent self-employed, 2025                                          

In [3]:
industry_cats = pd.read_excel('occupation.xlsx', sheet_name='Table 1.10', skiprows=1)
industry_cats = industry_cats[industry_cats['2025 National Employment Matrix code'].str.endswith('-0000')]
industry_cats = industry_cats[['2025 National Employment Matrix code', '2025 National Employment Matrix title']]
industry_cats = industry_cats.rename(columns={
    '2025 National Employment Matrix code': 'Industry code', '2025 National Employment Matrix title': 'Industry title'
})
industry_cats['Industry code'] = industry_cats['Industry code'].str.replace('-0000','', regex=False)

joined_df = pd.merge(joined_df, industry_cats, how='left', on='Industry code')
joined_df.head(2)
cleaned_df = (
    joined_df
    .replace({pd.NA: None, np.nan: None})
    .where(pd.notna(joined_df), None)
    .to_dict(orient='records')
    )

In [4]:
payload = {
    "raw_data": cleaned_df,
}

with open('../../public/data/bls_ai_exposure.json', 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, allow_nan=False)

In [ ]:
ai_exposure_rank_order = ['Very high', 'High', 'Moderate', 'Low']

joined_df['Relative AI exposure'] = pd.Categorical(joined_df['Relative AI exposure'], categories=ai_exposure_rank_order, ordered=True)
averages = joined_df.groupby('Relative AI exposure')['Employment change, percent'].agg(['mean', 'median']).reset_index()

In [ ]:
fig = px.scatter(
    joined_df,
    y='Employment change, percent',
    x='Total occupational separations rate, annual average',
    height=600,
    width=900,
    hover_data=['Occupation title'],
    title='AI Exposure vs Job Forecasted Change (%)',
    color='Relative AI exposure',
    category_orders={'Relative AI exposure': ai_exposure_rank_order},
)

fig.update_traces(
    marker=dict(
        size=8,
        opacity=0.7,
        line=dict(width=1, color='DarkSlateGrey')
    )
)

separation_median = joined_df['Total occupational separations rate, annual average'].median()

fig.add_vline(
    x=separation_median,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Median separation rate: {separation_median:.2f}",
    annotation_position="top left"
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black"
)

# fig.add_trace(
#     go.Scatter(
#         x=averages['mean'],
#         y=averages['Relative AI exposure'],
#         mode='text+markers',
#         marker=dict(symbol='diamond', size=10),
#         name='Average',
#     )
# )

# fig.add_trace(
#     go.Scatter(
#         x=averages['median'],
#         y=averages['Relative AI exposure'],
#         mode='text+markers',
#         marker=dict(symbol='diamond', size=10),
#         name='Median',
#     )
# )


fig.update_layout(
    coloraxis_showscale=False,
    template='plotly_white',
)
fig.show()